In [43]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [44]:
class LineRatePredictor(nn.Module):
    def __init__(self, num_features):
        super(LineRatePredictor, self).__init__()

        self.conv1 = nn.Conv1d(num_features, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=32,
            batch_first=True
        )

        self.fc1 = nn.Linear(32, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        # x shape: (batch, time, features)

        x = x.permute(0, 2, 1)       # → (batch, features, time)

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = x.permute(0, 2, 1)       # → (batch, time, channels)

        x, _ = self.lstm(x)

        x = x[:, -1, :]              # last timestep

        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x

In [45]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

In [72]:
class SensorDataset(Dataset):

    def __init__(self, df, window_size=20, shift=10):
        self.data = df.iloc[:,1:4].values.astype(np.float32)
        self.labels = df.iloc[:,-1].values.astype(np.float32)

        self.window = window_size
        self.shift = shift

    def __len__(self):
        return len(self.data) - self.window - self.shift

    def __getitem__(self, idx):

        X = self.data[idx : idx + self.window]

        y = self.labels[idx + self.window + self.shift]

        return torch.tensor(X), torch.tensor(y)
        
        

In [73]:
df = pd.read_csv("./sensor_dataset.csv")

dummy_set = SensorDataset(df)
dummy_loader = DataLoader(dummy_set, batch_size=32, shuffle=True)

In [74]:
model = LineRatePredictor(3)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [70]:
def train_model(model, train_loader, val_loader, epochs=50):

    for epoch in range(epochs):

        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            preds = model(X_batch).squeeze()

            loss = criterion(preds, y_batch)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                preds = model(X_batch).squeeze()

                loss = criterion(preds, y_batch)

                val_loss += loss.item()

                predicted = (preds > 0.5).float()

                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        accuracy = correct / total

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {accuracy:.4f}"
        )

In [75]:
train_model(model=model, train_loader=dummy_loader, val_loader=dummy_loader)

Epoch 1/50 | Train Loss: 0.2151 | Val Loss: 0.1683 | Val Acc: 0.9598
Epoch 2/50 | Train Loss: 0.1699 | Val Loss: 0.1680 | Val Acc: 0.9598
Epoch 3/50 | Train Loss: 0.1685 | Val Loss: 0.1694 | Val Acc: 0.9598
Epoch 4/50 | Train Loss: 0.1689 | Val Loss: 0.1684 | Val Acc: 0.9598
Epoch 5/50 | Train Loss: 0.1692 | Val Loss: 0.1685 | Val Acc: 0.9598
Epoch 6/50 | Train Loss: 0.1700 | Val Loss: 0.1682 | Val Acc: 0.9598
Epoch 7/50 | Train Loss: 0.1688 | Val Loss: 0.1698 | Val Acc: 0.9598
Epoch 8/50 | Train Loss: 0.1684 | Val Loss: 0.1688 | Val Acc: 0.9598
Epoch 9/50 | Train Loss: 0.1702 | Val Loss: 0.1689 | Val Acc: 0.9598
Epoch 10/50 | Train Loss: 0.1690 | Val Loss: 0.1691 | Val Acc: 0.9598
Epoch 11/50 | Train Loss: 0.1689 | Val Loss: 0.1684 | Val Acc: 0.9598
Epoch 12/50 | Train Loss: 0.1686 | Val Loss: 0.1682 | Val Acc: 0.9598
Epoch 13/50 | Train Loss: 0.1683 | Val Loss: 0.1710 | Val Acc: 0.9598
Epoch 14/50 | Train Loss: 0.1688 | Val Loss: 0.1684 | Val Acc: 0.9598
Epoch 15/50 | Train Loss: 0.1

In [ ]:
!pwd

/content
